📘 Prova Final AEDI - Questão 1: Regressão Linear Imobiliária
Contexto: Análise preditiva de preços de imóveis em King County (EUA). Metodologia: OLS (Ordinary Least Squares), Diagnóstico de Resíduos e Refinamento Log-Linear.

1. Preparação do Ambiente e Carga de Dados
A integridade dos dados é o primeiro passo de qualquer análise rigorosa. Removemos variáveis que não contribuem para a explicação estrutural do preço (id) ou que exigiriam tratamento de séries temporais (date), focando em uma análise cross-sectional (transversal).

In [2]:
# @title 1. Setup e Carga de Dados
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
px.defaults.template = "plotly_white"

# Carregamento do Arquivo Local
try:
    df = pd.read_csv('kc_house_data.csv')
    # Limpeza: Remoção de identificadores e data (foco nas características físicas/geográficas)
    df_clean = df.drop(['id', 'date'], axis=1)

    print(f"Base carregada: {df_clean.shape[0]} observações e {df_clean.shape[1]} atributos.")

    # Verificação de Qualidade de Dados (Requisito Acadêmico)
    null_check = df_clean.isnull().sum().sum()
    if null_check == 0:
        print("✅ Data Quality: Base íntegra, sem valores ausentes.")
    else:
        print(f"⚠️ Atenção: {null_check} valores ausentes detectados.")

    display(df_clean.head())

except Exception as e:
    print(f"Erro ao carregar o arquivo: {e}")

Base carregada: 21613 observações e 19 atributos.
✅ Data Quality: Base íntegra, sem valores ausentes.


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


### 2. Análise Descritiva dos Dados (20%)

Antes de modelar, precisamos investigar a distribuição da variável alvo (price). A inferência estatística clássica assume normalidade, e desvios significativos podem enviesar os estimadores.

Utilizaremos visualizações e medidas de forma (skewness e kurtosis) para diagnosticar a necessidade de transformações futuras.

In [3]:
# @title 2. Análise Exploratória (EDA)
# Análise de Momentos Estatísticos
desc_stats = df_clean[['price', 'sqft_living', 'grade']].describe().T
desc_stats['skewness'] = df_clean[['price', 'sqft_living', 'grade']].skew()
desc_stats['kurtosis'] = df_clean[['price', 'sqft_living', 'grade']].kurt()

print("--- Estatísticas Descritivas ---")
print("Nota: Skewness > 1 indica forte assimetria à direita (cauda longa).")
display(desc_stats[['mean', '50%', 'std', 'min', 'max', 'skewness']].style.background_gradient(cmap='Blues', subset=['skewness']))

# Visualização 1: Distribuição do Target
fig_hist = px.histogram(df_clean, x='price', nbins=50,
                       title='Distribuição de Preços: Assimetria Positiva',
                       labels={'price': 'Preço ($)'},
                       color_discrete_sequence=['#1f77b4'])
fig_hist.add_vline(x=df_clean['price'].mean(), line_dash="dash", line_color="red", annotation_text="Média")
fig_hist.add_annotation(x=4000000, y=100, text="Outliers (Imóveis de Luxo)", showarrow=True)
fig_hist.show()

# Visualização 2: O Fator Localização (Latitude vs Preço)
# Inspirado no notebook de referência: Latitude alta (Norte) tende a ter preços maiores.
fig_map = px.scatter_mapbox(df_clean, lat="lat", lon="long", color="price", size="sqft_living",
                  color_continuous_scale=px.colors.sequential.Jet, size_max=15, zoom=8.5,
                  title="Geo-Spatial Analysis: Concentração de Valor (Norte vs Sul)",
                  mapbox_style="carto-positron", height=600)
fig_map.show()

# Visualização 3: Correlações
corr_matrix = df_clean.corr()
fig_corr = px.imshow(corr_matrix, text_auto='.2f', aspect="auto", height=700,
                    title='Matriz de Correlação de Pearson',
                    color_continuous_scale='RdBu_r', origin='lower')
fig_corr.show()

--- Estatísticas Descritivas ---
Nota: Skewness > 1 indica forte assimetria à direita (cauda longa).


,mean,50%,std,min,max,skewness
price,540088.141767,450000.000000,367127.196483,75000.000000,7700000.000000,4.024069
sqft_living,2079.899736,1910.000000,918.440897,290.000000,13540.000000,1.471555
grade,7.656873,7.000000,1.175459,1.000000,13.000000,0.771103


Interpretação: A análise visual e estatística revela que a variável price possui uma forte assimetria positiva (skewness $\approx$ 4.02), caracterizada por uma cauda longa à direita (imóveis de altíssimo valor). Isso sugere que um modelo linear direto pode sofrer com a variância não constante. Além disso, o mapa geográfico confirma que a latitude é um discriminante crucial de valor.3.

### 3. Construção do Modelo de Regressão Linear (30%)

Para a primeira iteração (Baseline), utilizaremos o método dos Mínimos Quadrados Ordinários (OLS). Antes de treinar, realizaremos um teste de Multicolinearidade (VIF). Variáveis altamente correlacionadas (como sqft_living e sqft_above) podem inflar a variância dos coeficientes, tornando a interpretação de negócios instável.

In [4]:
# @title 3.1 Modelagem (V1 - Linear Puro)
# Definição de Features baseada na análise de correlação e literatura
target = 'price'
features = ['sqft_living', 'grade', 'bedrooms', 'bathrooms', 'view', 'lat', 'waterfront', 'floors', 'yr_built']

X = df_clean[features]
y = df_clean[target]

# Adição de constante para o intercepto (Beta_0)
X = sm.add_constant(X)

# Divisão Treino/Teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# [Diagnóstico Pré-Modelo] Multicolinearidade (VIF)
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print("--- Teste de Multicolinearidade (VIF) ---")
print("Critério: VIF > 5 requer atenção; VIF > 10 é crítico.")
display(vif_data.sort_values(by="VIF", ascending=False).style.background_gradient(cmap='Reds'))

# Treinamento do Modelo OLS
model_v1 = sm.OLS(y_train, X_train).fit()

# Exibição dos Resultados
print(model_v1.summary())

--- Teste de Multicolinearidade (VIF) ---
Critério: VIF > 5 requer atenção; VIF > 10 é crítico.


,Feature,VIF
0,const,149392.449118
1,sqft_living,4.072705
4,bathrooms,3.167157
2,grade,2.990810
9,yr_built,1.738169
3,bedrooms,1.627878
8,floors,1.551205
5,view,1.345414
7,waterfront,1.196922
6,lat,1.085786


                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.691
Model:                            OLS   Adj. R-squared:                  0.691
Method:                 Least Squares   F-statistic:                     3759.
Date:                Wed, 19 Nov 2025   Prob (F-statistic):               0.00
Time:                        20:52:55   Log-Likelihood:            -2.0620e+05
No. Observations:               15129   AIC:                         4.124e+05
Df Residuals:                   15119   BIC:                         4.125e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const       -2.074e+07   6.33e+05    -32.787      

### 4. Interpretação e Diagnóstico de Pressupostos (10%)

O $R^2$ pode ser alto, mas se os resíduos violarem os pressupostos de Gauss-Markov, os testes de hipótese (p-valor) perdem a validade. Realizamos aqui uma "auditoria estatística" do modelo.Pressupostos Avaliados:Normalidade dos Resíduos: Teste Jarque-Bera.Homocedasticidade: Teste Breusch-Pagan (variância constante dos erros).

In [5]:
# @title 4. Auditoria de Resíduos (Diagnóstico)
residuals = model_v1.resid

# 1. Testes Estatísticos Formais
jb_test = sms.jarque_bera(residuals)
bp_test = sms.het_breuschpagan(residuals, model_v1.model.exog)

print(f"--- Diagnóstico de Pressupostos ---")
print(f"1. Normalidade (Jarque-Bera): p-valor = {jb_test[1]:.4e}")
print(f"   >> {'H0 Rejeitada (Não Normal)' if jb_test[1] < 0.05 else 'Normalidade Aceita'}")
print(f"2. Homocedasticidade (Breusch-Pagan): p-valor = {bp_test[1]:.4e}")
print(f"   >> {'H0 Rejeitada (Heterocedasticidade Presente)' if bp_test[1] < 0.05 else 'Homocedasticidade Aceita'}")

# 2. Diagnóstico Visual
fig_diag = make_subplots(rows=1, cols=2, subplot_titles=("Resíduos vs Ajustados (Homocedasticidade)", "Q-Q Plot (Normalidade)"))

# Scatter: Resíduos vs Fitted
fig_diag.add_trace(go.Scatter(x=model_v1.fittedvalues, y=residuals, mode='markers', marker=dict(opacity=0.5, size=4), name='Resíduos'), row=1, col=1)
fig_diag.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

# Q-Q Plot Manual
qq_theory, qq_sample = stats.probplot(residuals, dist="norm")[0]
fig_diag.add_trace(go.Scatter(x=qq_theory, y=qq_sample, mode='markers', name='Q-Q'), row=1, col=2)
fig_diag.add_trace(go.Scatter(x=[min(qq_theory), max(qq_theory)], y=[min(qq_theory), max(qq_theory)], mode='lines', line=dict(color='red'), showlegend=False), row=1, col=2)

fig_diag.update_layout(title='Diagnóstico de Falhas do Modelo Linear (V1)', height=500)
fig_diag.show()

--- Diagnóstico de Pressupostos ---
1. Normalidade (Jarque-Bera): p-valor = 0.0000e+00
   >> H0 Rejeitada (Não Normal)
2. Homocedasticidade (Breusch-Pagan): p-valor = 0.0000e+00
   >> H0 Rejeitada (Heterocedasticidade Presente)


**Análise dos Resultados:** O modelo apresenta violações críticas.

Heterocedasticidade: O gráfico da esquerda mostra uma clara forma de "cone" (a variância do erro aumenta conforme o preço aumenta). O teste Breusch-Pagan confirma isso (p < 0.05).

Não-Normalidade: O Q-Q Plot desvia drasticamente nas caudas, indicando que o modelo falha em capturar a dinâmica dos imóveis de luxo e dos muito baratos.

### 5. Ajustes no Modelo (Refinamento Log-Linear) (30%)

Para corrigir a heterocedasticidade e a distribuição assimétrica, aplicaremos a transformação Logarítmica na variável resposta (price). Esta técnica lineariza crescimentos exponenciais e estabiliza a variância, comum em dados financeiros e imobiliários.

Modelo V2: $\log(Price) = \beta_0 + \beta_1 X_1 + ... + \epsilon$

In [6]:
# @title 5. Refinamento (Modelo Log-Linear)
# Transformação Logarítmica
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

# Treinamento V2
model_v2 = sm.OLS(y_train_log, X_train).fit()

# Previsões e Reversão para Comparação Real
y_pred_v1 = model_v1.predict(X_test)
y_pred_v2_log = model_v2.predict(X_test)
y_pred_v2 = np.exp(y_pred_v2_log) # Revertendo log para $

# Comparativo de Métricas
metrics = pd.DataFrame({
    'Métrica': ['R² (Ajuste)', 'RMSE (Erro Médio $)', 'Condição dos Resíduos'],
    'Modelo V1 (Linear)': [f"{model_v1.rsquared:.3f}", f"${np.sqrt(mean_squared_error(y_test, y_pred_v1)):,.2f}", "Heterocedásticos (Cone)"],
    'Modelo V2 (Log-Linear)': [f"{model_v2.rsquared:.3f}", f"${np.sqrt(mean_squared_error(y_test, y_pred_v2)):,.2f}", "Mais Estáveis"]
})

print("--- Resultados do Ajuste ---")
display(metrics)

# Visualização da Melhoria
fig_v2 = px.scatter(x=model_v2.fittedvalues, y=model_v2.resid,
                   title="Resíduos vs Ajustados (Modelo V2 Log) - Estabilização da Variância",
                   labels={'x': 'Log(Price) Previsto', 'y': 'Resíduos'}, opacity=0.3)
fig_v2.add_hline(y=0, line_color="red", line_dash="dash")
fig_v2.show()

print("\n--- Summary do Modelo Ajustado ---")
print(model_v2.summary())

--- Resultados do Ajuste ---


,Métrica,Modelo V1 (Linear),Modelo V2 (Log-Linear)
0,R² (Ajuste),0.691,0.758
1,RMSE (Erro Médio $),"$212,301.27","$291,438.21"
2,Condição dos Resíduos,Heterocedásticos (Cone),Mais Estáveis



--- Summary do Modelo Ajustado ---
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.758
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     5248.
Date:                Wed, 19 Nov 2025   Prob (F-statistic):               0.00
Time:                        20:56:21   Log-Likelihood:                -990.08
No. Observations:               15129   AIC:                             2000.
Df Residuals:                   15119   BIC:                             2076.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const         

### 6. Tomada de Decisão e Storytelling de Negócios (10%)

Com o modelo Log-Linear, a interpretação dos coeficientes muda para elasticidade/variação percentual.

Fórmula: $\Delta\% \approx (e^{\beta} - 1) \times 100$

Isso nos permite fornecer insights estratégicos quantificados para a diretoria.

In [9]:
# @title 6. Inteligência de Negócios (Interpretação Estratégica)
# Calculando impacto percentual exato
impact_df = pd.DataFrame({
    'Fator': model_v2.params.index,
    'Coeficiente': model_v2.params.values,
    'Impacto_Percentual': (np.exp(model_v2.params.values) - 1) * 100
})

# Filtrando constantes e ordenando por impacto positivo
biz_insights = impact_df[impact_df['Fator'] != 'const'].sort_values(by='Impacto_Percentual', ascending=False)

# Definir 'Fator' como índice para acesso fácil usando .loc
biz_insights = biz_insights.set_index('Fator')

# Visualização Executiva
fig_biz = px.bar(biz_insights, x='Impacto_Percentual', y=biz_insights.index, orientation='h',
                title='<b>Drivers de Valorização Imobiliária:</b> Onde Alocar Capital?',
                text_auto='.1f',
                labels={'Impacto_Percentual': 'Valorização Estimada (%)'},
                color='Impacto_Percentual', color_continuous_scale='Viridis')
fig_biz.update_layout(height=600, showlegend=False)
fig_biz.add_vline(x=0, line_color='black')
fig_biz.show()

# Geração de Texto Dinâmico Baseado nos Dados Reais
lat_effect = biz_insights.loc['lat', 'Impacto_Percentual']
grade_effect = biz_insights.loc['grade', 'Impacto_Percentual']
water_effect = biz_insights.loc['waterfront', 'Impacto_Percentual']
age_effect = biz_insights.loc['yr_built', 'Impacto_Percentual']

print("--- RELATÓRIO DE ESTRATÉGIA DE INVESTIMENTO (Baseado em Dados Reais) ---")
print(f"\n1. A GEOGRAFIA É SOBERANA (Latitude):")
print(f"   O modelo indica um aumento exponencial de {lat_effect:.1f}% no preço para cada grau de latitude ao Norte.")
print(f"   Decisão: Focar aquisições na zona Norte de King County. Um imóvel idêntico no Sul vale drasticamente menos.")

print(f"\n2. O PRÊMIO DE QUALIDADE (Grade):")
print(f"   Aumentar o padrão construtivo (Grade) em 1 unidade gera uma valorização média de {grade_effect:.1f}%.")
print(f"   Decisão: O ROI de reformas de alto padrão (ex: acabamentos de luxo) supera expansões de área simples.")

print(f"\n3. ESCASSEZ GERA VALOR (Waterfront):")
print(f"   Ter frente para a água adiciona um prêmio de aproximadamente {water_effect:.1f}% ao imóvel, ceteris paribus.")
print(f"   Decisão: Priorizar ativos costeiros mesmo que necessitem de reforma total, pois a localização é irreplicável.")

print(f"\n4. IMÓVEIS NOVOS (Year Built):")
print(f"   O coeficiente negativo ({age_effect:.3f}%) sugere que a idade do imóvel penaliza levemente o preço.")
print(f"   Decisão: Imóveis antigos exigem desconto na compra para compensar a depreciação percebida pelo mercado.")

--- RELATÓRIO DE ESTRATÉGIA DE INVESTIMENTO (Baseado em Dados Reais) ---

1. A GEOGRAFIA É SOBERANA (Latitude):
   O modelo indica um aumento exponencial de 279.3% no preço para cada grau de latitude ao Norte.
   Decisão: Focar aquisições na zona Norte de King County. Um imóvel idêntico no Sul vale drasticamente menos.

2. O PRÊMIO DE QUALIDADE (Grade):
   Aumentar o padrão construtivo (Grade) em 1 unidade gera uma valorização média de 19.4%.
   Decisão: O ROI de reformas de alto padrão (ex: acabamentos de luxo) supera expansões de área simples.

3. ESCASSEZ GERA VALOR (Waterfront):
   Ter frente para a água adiciona um prêmio de aproximadamente 45.6% ao imóvel, ceteris paribus.
   Decisão: Priorizar ativos costeiros mesmo que necessitem de reforma total, pois a localização é irreplicável.

4. IMÓVEIS NOVOS (Year Built):
   O coeficiente negativo (-0.383%) sugere que a idade do imóvel penaliza levemente o preço.
   Decisão: Imóveis antigos exigem desconto na compra para compensar a dep

## 📑 Principais termos do dataset traduzidos

| **Termo original** | **Tradução para o Brasil** |
|--------------------|-----------------------------|
| `sqft_living`      | Área útil interna da residência (m²) |
| `sqft_above`       | Área acima do nível térreo (m²) |
| `sqft_basement`    | Área do porão (m²) |
| `sqft_lot`         | Área total do terreno (m²) |
| `bathrooms`        | Número de banheiros |
| `bedrooms`         | Número de quartos |
| `floors`           | Número de pavimentos/andares |
| `waterfront`       | Frente para o mar/lago |
| `view`             | Qualidade da vista |
| `condition`        | Estado geral da casa |
| `grade`            | Padrão de construção/acabamento |
